In [1]:
# Installer les dépendances indiquées par le projet.
%pip install -r requirements.txt


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Chargement du corpus
from classes.Corpus import Corpus
from classes.SearchEngine import SearchEngine

corpus = Corpus("Animaux")
corpus.load("corpus_objets.csv")  
print(corpus)

Corpus 'Animaux' : 154 document(s), 149 auteur(s)


In [3]:
# Instanciation et vérification
moteur = SearchEngine(corpus)
moteur.search("animal behavior", n=3)

Construction de la matrice: 100%|██████████| 154/154 [00:00<00:00, 15277.99it/s]


,id_document,titre,auteur,source,score,url
0,107,Animal Behavior Analysis Methods Using Deep Le...,Edoardo Fazzari,arxiv,0.273335,http://arxiv.org/abs/2405.14002v1
1,134,Models of Animal Behavior as Active Particle S...,Amir Haluts,arxiv,0.210560,http://arxiv.org/abs/2401.14850v2
2,128,Collective Animal Behavior from Bayesian Estim...,Alfonso Pérez-Escudero,arxiv,0.180076,http://arxiv.org/abs/1105.1117v2


In [4]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd
from html import escape
from urllib.parse import urlparse
from pathlib import Path
pd.set_option("display.max_colwidth", None)

# Titre en HTML
titre = widgets.HTML(
    value="<h2 style='color:#3e2f23; margin:4px 0;'>Moteur de recherche</h2>"
)
champ_mots = widgets.Text(
    value="",
    placeholder="Entrez vos mots-clés",
    description="Mots clés :",
    layout=widgets.Layout(width="100%"),
)
curseur = widgets.IntSlider(
    value=5, min=1, max=20, step=1,
    description="Nb résultats :",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "90px"},
)
curseur_contexte = widgets.IntSlider(
    value=30, min=10, max=80, step=5,
    description="Contexte :",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "90px"},
)
filtre_source = widgets.Dropdown(
    options=["Toutes", "reddit", "arxiv"],
    value="Toutes",
    description="Source :",
    layout=widgets.Layout(width="100%"),
)

aide = widgets.HTML(
    "<p style='color:#3e2f23'>Source : recherche, concordancier et évolution. "
    "Contexte : concordancier uniquement. Évolution : premier mot nettoyé, "
    "occurrences brutes par mois de publication.</p>"
)

# La zone de résultats
sortie = widgets.Output()
sortie.add_class("zone-resultats")

# Images décoratives à gauche et à droite (n'importe quel PNG/JPG du projet)
octets_image = Path("chat.png").read_bytes() if Path("chat.png").is_file() else b""

image_gauche = widgets.Image(
    value=octets_image, format="png",
    layout=widgets.Layout(width="110px", flex="0 0 110px"),
)
image_droite = widgets.Image(
    value=octets_image, format="png",
    layout=widgets.Layout(width="110px", flex="0 0 110px"),
)


In [5]:
# Thème taupe
styles = HTML("""
<style>
    /* Les paddings et bordures sont inclus dans les largeurs */
    .moteur,
    .moteur * {
        box-sizing: border-box;
    }

    .moteur {
        background-color: #d8cfbf;       /* taupe clair */
        border: 2px solid #b3a68e;
        border-radius: 14px;
        padding: 16px;
        width: 100%;
        max-width: 100%;
        min-width: 0;
        margin: 0;
        overflow: visible;
    }

    /* Empêcher les widgets de dépasser leur conteneur */
    .moteur .jupyter-widgets {
        min-width: 0;
        max-width: 100%;
        margin: 0;
    }

    .moteur .widget-box,
    .moteur .widget-output,
    .moteur .widget-html {
        overflow: visible;
    }

    .moteur .widget-html-content {
        min-width: 0;
        max-width: 100%;
        overflow-wrap: anywhere;
    }

    /* Boutons et champs */
    .moteur .widget-button {
        border-radius: 10px;
        font-weight: bold;
        background-color: #6d5843 !important;
        color: #f5efe4 !important;
    }

    .moteur .widget-text input {
        border-radius: 8px;
    }

    .moteur .widget-label,
    .moteur .widget-readout {
        color: #3e2f23;
    }

    .exploration {
        border-left: 1px solid #a89a80;
        padding-left: 16px;
    }

    .moteur .actions-recherche {
        gap: 10px;
        flex-wrap: wrap;
    }

    .moteur .exploration {
        gap: 10px;
    }

    /* Zone de résultats */
    .zone-resultats {
        background-color: #c2b49b;       /* taupe plus sombre */
        color: #3e2f23;
        border-radius: 10px;
        padding: 10px;
    }

    .zone-resultats .output_area,
    .zone-resultats .output_subarea,
    .zone-resultats .jp-OutputArea,
    .zone-resultats .jp-OutputArea-child,
    .zone-resultats .jp-OutputArea-output {
        min-width: 0;
        max-width: 100%;
        overflow: visible;
    }

    /* Tableaux : largeur limitée et retour à la ligne */
    .zone-resultats table {
        width: 100%;
        table-layout: fixed;
    }

    .zone-resultats table,
    .zone-resultats table th,
    .zone-resultats table td {
        background-color: #c2b49b !important;
        color: #3e2f23 !important;
        border-color: #a89a80 !important;
    }

    .zone-resultats table th,
    .zone-resultats table td {
        white-space: normal;
        overflow-wrap: anywhere;
    }

    /* Textes longs et liens */
    .zone-resultats pre {
        white-space: pre-wrap;
        overflow-wrap: anywhere;
        max-width: 100%;
        color: #3e2f23;
        background: transparent;
    }

    .zone-resultats a {
        color: #3e2f23;
        text-decoration: underline;
    }

    /* Graphiques adaptés à la largeur disponible */
    .zone-resultats img,
    .zone-resultats svg {
        max-width: 100%;
        height: auto;
    }

    /* Bloc de lecture séparé du tableau */
.moteur .lecteur {
    background-color: #d8cfbf;
    border: 1px solid #a89a80;
    border-radius: 12px;
    padding: 16px;
    margin-top: 16px;
    gap: 12px;
}

/* Ligne du menu Lire */
.moteur .lecteur .widget-dropdown {
    height: auto;
    min-height: 36px;
    align-items: center;
}

.moteur .lecteur .widget-label {
    color: #3e2f23;
    font-weight: bold;
}

/* Menu déroulant assorti au thème */
.moteur .lecteur select {
    background-color: #f0e8da;
    color: #3e2f23;
    border: 1px solid #a89a80;
    border-radius: 8px;
    padding: 8px;
    min-width: 0;
    font-family: inherit;
}

.moteur .lecteur select:focus {
    outline: 2px solid #6d5843;
    outline-offset: 2px;
}

/* Texte du document */
.moteur .lecteur .widget-output {
    background-color: #c2b49b;
    color: #3e2f23;
    border-radius: 8px;
    padding: 12px;
}

.moteur .lecteur h3 {
    color: #3e2f23;
    margin: 0 0 12px;
    overflow-wrap: anywhere;
}

.moteur .lecteur pre {
    font-family: inherit;
    font-size: 14px;
    line-height: 1.6;
}

</style>
""")
display(styles)

In [6]:
MOTS_VIDES = {
    "that", "with", "from", "have", "their", "these", "those", "when",
    "where", "this", "what", "which", "will", "would", "there", "been",
    "were", "they", "them", "then", "than", "your", "into", "some",
    "just", "like", "also", "about", "after", "because", "while",
    "more", "other", "such", "only", "over", "much", "here", "well",
    "the", "and", "she", "but", "for", "are", "her", "was", "our",
    "him", "can", "not", "has", "his", "you"
}

def afficher_lecteur(tableau):
    # Une liste de documents, même si plusieurs occurrences proviennent du même texte.
    ids = tableau["id_document"].drop_duplicates().tolist()
    choix = widgets.Dropdown(
        options=[("Choisir un document à lire", None)] + [
            (f"{identifiant} : {corpus.documents[identifiant].titre}", identifiant)
            for identifiant in ids
        ],
        value=None,
        description="Lire :",
        layout=widgets.Layout(width="100%"),
    )
    lecture = widgets.Output(layout=widgets.Layout(width="100%"))

    def lire_document(changement):
        with lecture:
            lecture.clear_output()
            identifiant = changement["new"]
            if identifiant is None:
                return
            doc = corpus.documents[identifiant]
            display(HTML("<h3>" + escape(str(doc.titre)) + "</h3>"))
            print(f"Auteur : {doc.auteur} | Source : {doc.getType()} | Date : {doc.date}")
            if doc.getType() == "arxiv":
                print("Le corpus contient le résumé, pas le texte intégral de l'article.")
            # Échapper le texte empêche un document d'injecter du HTML.
            display(HTML("<pre>" + escape(str(doc.texte)) + "</pre>"))
            url = str(doc.url)
            if urlparse(url).scheme in {"http", "https"} and urlparse(url).netloc:
                display(HTML(
                    '<a href="' + escape(url, quote=True)
                    + '" target="_blank" rel="noopener noreferrer">'
                    "Ouvrir le document original</a>"
                ))
            else:
                print("Lien d'origine indisponible.")

    choix.observe(lire_document, names="value")
    lecteur = widgets.VBox(
        [choix, lecture],
        layout=widgets.Layout(width="100%"),
    )
    lecteur.add_class("lecteur")
    display(lecteur)

def clique_bouton(b):
    with sortie:
        sortie.clear_output()
        if not champ_mots.value.strip():
            print("Entrez au moins un mot-clé.")
            return
        resultats = moteur.search(
            champ_mots.value,
            n=curseur.value,
            source=filtre_source.value,
        )
        if resultats.empty:
            print("Aucun document trouvé.")
        else:
            print(f"{len(resultats)} document(s) trouvés.")
            display(resultats.style.format({"score": "{:.6g}"}, escape="html"))
            afficher_lecteur(resultats)

def clique_concorde(b):
    with sortie:
        sortie.clear_output()
        if not champ_mots.value.strip():
            print("Entrez un mot pour le concordancier.")
            return
        mot = champ_mots.value.split()[0]
        tableau = corpus.concorde(
            mot, taille_contexte=curseur_contexte.value, source=filtre_source.value
        )
        if tableau.empty:
            print(f"Aucune occurrence de « {mot} ».")
        else:
            print(
                f"{len(tableau)} occurrence(s) de « {mot} » dans "
                f"{tableau['id_document'].nunique()} document(s)."
            )
            affiches = tableau.head(curseur.value)  # nb d'occurrences = curseur Nb résultats
            print(f"Affichage des {len(affiches)} premières occurrences.")
            display(affiches)
            afficher_lecteur(affiches)

def clique_stats(b):
    with sortie:
        sortie.clear_output()
        print("Statistiques globales : les mots-clés et le filtre source ne sont pas utilisés.")
        freq = corpus.construire_freq()
        top = (
            freq[~freq["mot"].isin(MOTS_VIDES) & (freq["mot"].str.len() > 2)]
            .sort_values("term frequency", ascending=False)
            .head(15)
        )
        fig, ax = plt.subplots(figsize=(9, 4), facecolor="#d8cfbf")
        ax.set_facecolor("#c2b49b")
        plt.bar(top["mot"], top["term frequency"], color="#6d5843")
        plt.xticks(rotation=45, ha="right", color="#3e2f23")
        plt.title("Les 15 mots les plus fréquents du corpus", color="#3e2f23")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

def clique_evolution(b):
    with sortie:
        sortie.clear_output()
        if not champ_mots.value.strip():
            print("Entrez un mot pour suivre son évolution.")
            return
        # Occurrences du mot regroupées par le mois de publication.
        # La méthode partagée normalise le mot et ignore les dates mal formées.
        evolution = corpus.evolution_mot(champ_mots.value, source=filtre_source.value)
        if evolution.empty:
            print("Aucune occurrence exploitable dans cette source (dates valides requises).")
            return
        mot = corpus.nettoyer_texte(champ_mots.value).split()[0]
        fig, ax = plt.subplots(figsize=(9, 4), facecolor="#d8cfbf")
        ax.set_facecolor("#c2b49b")
        ax.plot(pd.to_datetime(evolution["periode"]), evolution["occurrences"],
                marker="o", color="#6d5843")
        ax.set_title(f"Évolution du mot « {mot} » : {filtre_source.value}", color="#3e2f23")
        ax.set_ylabel("Occurrences dans le corpus", color="#3e2f23")
        ax.tick_params(colors="#3e2f23")
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
        plt.close(fig)


import math

def clique_comparer(b):
    with sortie:
        sortie.clear_output()

        print("Comparaison globale des deux sources : les champs de recherche ne sont pas utilisés.")
        # TF de chaque sous-corpus : chaque source = un gros document
        tf = {"reddit": {}, "arxiv": {}}
        for doc in corpus.documents.values():
            source = doc.getType()
            if source not in tf:
                continue
            for mot in corpus.nettoyer_texte(doc.texte).split():
                tf[source][mot] = tf[source].get(mot, 0) + 1

        # TF-IDF : idf = log(2/df), df = nombre de sous-corpus contenant le mot
        # Un mot présent des deux côtés a un idf nul : il disparaît.
        scores = {"reddit": {}, "arxiv": {}}
        for source in tf:
            for mot, nb in tf[source].items():
                if mot.isalpha() and len(mot) > 3 and mot not in MOTS_VIDES:      # écarte les mots courts et les mots vides
                    df = sum(mot in tf[s] for s in tf)  # 1 ou 2
                    if df == 1:
                        scores[source][mot] = nb * math.log(2)

        # Deux graphiques côte à côte
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor="#d8cfbf")
        for axe, source in zip(axes, ["reddit", "arxiv"]):
            top = sorted(scores[source].items(),
                         key=lambda x: x[1], reverse=True)[:12]
            if top:
                mots_top = [m for m, _ in top][::-1]
                valeurs = [v for _, v in top][::-1]
                axe.barh(mots_top, valeurs, color="#6d5843")
            else:
                axe.text(0.5, 0.5, "Aucun mot spécifique", ha="center",
                         transform=axe.transAxes)
            axe.set_title(f"Mots spécifiques — {source}", color="#3e2f23")
            axe.tick_params(colors="#3e2f23")
            axe.set_facecolor("#c2b49b")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Synthèse chiffrée
        communs = set(tf["reddit"]) & set(tf["arxiv"])
        print(f"Mots communs aux deux sources : {len(communs)}")
        print(f"Mots propres à Reddit : {len(set(tf['reddit']) - set(tf['arxiv']))}")
        print(f"Mots propres à arXiv : {len(set(tf['arxiv']) - set(tf['reddit']))}")

In [7]:
bouton = widgets.Button(description="Rechercher")
bouton.on_click(clique_bouton)

bouton_concorde = widgets.Button(description="Concordancier")
bouton_concorde.on_click(clique_concorde)

bouton_stats = widgets.Button(description="Statistiques")
bouton_stats.on_click(clique_stats)
bouton_evolution = widgets.Button(description="Évolution")
bouton_evolution.on_click(clique_evolution)

bouton_comparer = widgets.Button(description="Comparer")
bouton_comparer.on_click(clique_comparer)

ligne_boutons = widgets.HBox(
    [bouton, bouton_concorde, bouton_evolution],
    layout=widgets.Layout(width="100%", flex_flow="row wrap"),
)

ligne_boutons.add_class("actions-recherche")

# Colonne centrale : titre, champ, curseur, boutons
colonne = widgets.VBox(
    [titre, champ_mots, curseur, filtre_source, curseur_contexte, aide, ligne_boutons],
    layout=widgets.Layout(flex="1 1 360px", min_width="0", padding="0 16px"),
)

# Actions globales, indépendantes des champs de recherche.
exploration = widgets.VBox(
    [
        widgets.HTML("<h3 style='color:#3e2f23'>Explorer le corpus</h3>"),
        widgets.HTML("<p style='color:#3e2f23'>Corpus entier,<br>sans les champs de recherche.</p>"),
        bouton_stats, bouton_comparer,
    ],
    layout=widgets.Layout(flex="0 1 200px", min_width="160px", margin="16px 8px"),
)
exploration.add_class("exploration")

# Images à gauche et à droite de cette colonne
ligne_principale = widgets.HBox(
    [image_gauche, colonne, exploration, image_droite],
    layout=widgets.Layout(width="100%", align_items="center", flex_flow="row wrap"),
)

# Zone de résultats sous le tout
sortie.layout.width = "100%"
interface = widgets.VBox(
    [ligne_principale, sortie], layout=widgets.Layout(width="100%")
)
interface.add_class("moteur")

display(interface)
